# CS781 Phase 2 — Retention-lite model


## 0. Write `lner.py`, `lner_ret.py`, `ensemble.py`

In [1]:
%%writefile lner.py
#!/usr/bin/env python3
"""
lner.py  --  CS781 Phase 2 (Legal NER) pipeline on top of YOUR Phase-1 RoPE+MQA+SwiGLU decoder.

One run = one adaptation method.  Examples:
  python lner.py --method full  --attn bidir  --out runs/full_bidir
  python lner.py --method lorma --out runs/lorma
  python lner.py --method kd --teacher runs/full_bidir/best.pt --out runs/kd

Sections
  A. Phase-1 tokenizer + model code (copied from your notebooks; the ONLY edits are marked  # [P2-EDIT])
  B. Offset-exact tokenisation, BIO labelling, windowing
  C. Adaptation methods (full / partial / bitfit / lora / lorma / lorma_pi / adapter / ia3 / kd)
  D. Training, decoding (greedy / BIO-constrained Viterbi), strict evaluation, submission writer
"""
import argparse, hashlib, json, math, os, pickle, random, re, sys, time, zipfile
from bisect import bisect_left
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IGN = -100
PAD_ID = 256          # "<pad>" in your tokenizer (256 + index in SPECIAL_TOKENS)
EOS_ID = 259          # "</s>"  (word separator in your Phase-1 tokenizer)


class ByteLevelBPETokenizer:
    SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            tid = 256 + i
            self.special_to_id[tok] = tid
            self.id_to_bytes[tid] = tok.encode("utf-8")

    @property
    def vocab_size(self):
        return len(self.id_to_bytes)

    @staticmethod
    def _pair_counts(sequences):
        counts = {}
        for seq, freq in sequences:
            for pair in zip(seq, seq[1:]):
                counts[pair] = counts.get(pair, 0) + freq
        return counts

    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                out.append(new_id); i += 2
            else:
                out.append(seq[i]); i += 1
        return out

    def train(self, text, vocab_size, max_words=100_000, verbose=False):   # only used by the smoke test
        num_merges = vocab_size - self.vocab_size
        word_freq = {}
        for word in text.split():
            word_freq[word] = word_freq.get(word, 0) + 1
        sequences = [[list(w.encode("utf-8")), f] for w, f in word_freq.items()]
        next_id = self.vocab_size
        for _ in range(num_merges):
            counts = self._pair_counts(sequences)
            if not counts: break
            best = max(counts, key=counts.get)
            self.merges[best] = next_id
            self.id_to_bytes[next_id] = self.id_to_bytes[best[0]] + self.id_to_bytes[best[1]]
            for e in sequences: e[0] = self._apply_merge(e[0], best, next_id)
            next_id += 1
        return self

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": {f"{a},{b}": n for (a, b), n in self.merges.items()}}, f)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1 = x[..., ::2]; x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, q, k):
        T = q.size(-2)
        cos = self.cos[:, :, :T, :]; sin = self.sin[:, :, :T, :]
        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class MultiQueryCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % kv_heads == 0
        self.n_heads, self.kv_heads = n_heads, kv_heads
        self.d_head = d_model // n_heads
        self.kv_dim = kv_heads * self.d_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.d_head, max_seq_len=max_seq_len, base=rope_base)
        self.dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.causal = True                                   # [P2-EDIT] switch for bidirectional fine-tuning

    def forward(self, x, pad_mask=None):                     # [P2-EDIT] optional key padding mask
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        q, k = self.rope(q, k)
        if self.kv_heads != self.n_heads:
            r = self.n_heads // self.kv_heads
            k = k.repeat_interleave(r, dim=1); v = v.repeat_interleave(r, dim=1)
        dp = self.dropout if self.training else 0.0
        if self.causal:                                      # right padding is harmless under a causal mask
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=dp, is_causal=True)
        else:
            m = None if pad_mask is None else pad_mask[:, None, None, :]   # True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, dropout_p=dp, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.silu(self.gate(x)) * self.up(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.attn = MultiQueryCausalAttention(d_model, n_heads, kv_heads, max_seq_len, dropout, rope_base)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, pad_mask=None):                     # [P2-EDIT] pass the mask through
        x = x + self.attn(self.norm_attn(x), pad_mask)
        x = x + self.ffn(self.norm_ffn(x))
        return x


class RoPEMQADecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, kv_heads, d_ff, max_seq_len,
                 dropout=0.1, rope_base=10000.0, tie_weights=True):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base)
                                     for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_emb.weight


DEFAULT_LM_CFG = dict(vocab_size=12000, d_model=768, n_layers=12, n_heads=12, kv_heads=1,
                      d_ff=3072, max_seq_len=512, dropout=0.1, rope_base=10000.0)


def build_lm(cfg):
    c = {**DEFAULT_LM_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_LM_CFG}}
    return RoPEMQADecoderLM(**c, tie_weights=True), c


def load_phase1(path, n_layers=None, layer_map=None):
    """Load your Phase-1 weights.  Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config})
    or a raw state_dict (model_stepXXXXX.pt).  Optionally keeps only `layer_map` blocks (for the KD student)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], ck.get("config", {})
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]
        rc = ck.get("run_config", {})
        cfg = dict(vocab_size=rc.get("VOCAB_SIZE"), d_model=rc.get("D_MODEL"), n_layers=rc.get("N_LAYERS"),
                   n_heads=rc.get("N_HEADS"), d_ff=rc.get("D_FF"), max_seq_len=rc.get("BLOCK_SIZE"),
                   kv_heads=rc.get("KV_HEADS"), rope_base=rc.get("ROPE_BASE"))
        cfg = {k: v for k, v in cfg.items() if v is not None}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    cfg["n_layers"] = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["blocks.0.ffn.gate.weight"].shape[0]
    cfg["kv_heads"] = sd["blocks.0.attn.k_proj.weight"].shape[0] // (cfg["d_model"] // cfg.get("n_heads", 12))
    if layer_map is not None:                                # layer-dropped student: blocks.{old} -> blocks.{new}
        new = {k: v for k, v in sd.items() if not k.startswith("blocks.")}
        for ni, oi in enumerate(layer_map):
            for k, v in sd.items():
                if k.startswith(f"blocks.{oi}."):
                    new[f"blocks.{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_layers"] = new, len(layer_map)
    lm, cfg = build_lm(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k != "lm_head.weight"] + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg


WORD_RE = re.compile(r"\S+")             # same segmentation as str.split() used in Phase 1
PIECE_RE = re.compile(r"\w+|[^\w\s]")    # punctuation is split off so entity edges can sit next to it


class Piecer:
    """BPE-encodes one piece at a time and returns char offsets for every BPE token (cached)."""
    def __init__(self, tok):
        self.tok, self.cache = tok, {}

    def piece(self, s):
        r = self.cache.get(s)
        if r is None:
            tok = self.tok
            seq = list(s.encode("utf-8"))
            while len(seq) > 1:
                mergeable = [p for p in zip(seq, seq[1:]) if p in tok.merges]
                if not mergeable: break
                pair = min(mergeable, key=lambda p: tok.merges[p])
                seq = tok._apply_merge(seq, pair, tok.merges[pair])
            b2c = []
            for ci, ch in enumerate(s):
                b2c.extend([ci] * len(ch.encode("utf-8")))
            offs, b = [], 0
            for t in seq:
                n = len(tok.id_to_bytes[t])
                offs.append((b2c[b], b2c[b + n - 1] + 1)); b += n
            r = (tuple(seq), tuple(offs))
            self.cache[s] = r
        return r

    def encode_doc(self, text):
        ids, cs, ce = [], [], []
        for wm in WORD_RE.finditer(text):
            w, ws = wm.group(), wm.start()
            for pm in PIECE_RE.finditer(w):
                tids, offs = self.piece(pm.group())
                base = ws + pm.start()
                for t, (a, b) in zip(tids, offs):
                    ids.append(t); cs.append(base + a); ce.append(base + b)
            ids.append(EOS_ID); cs.append(-1); ce.append(-1)      # word separator, exactly as in pre-training
        return np.array(ids, np.int32), np.array(cs, np.int32), np.array(ce, np.int32)


class LabelSpace:
    def __init__(self, labels):
        self.labels = list(labels)
        self.k = {l: i for i, l in enumerate(self.labels)}
        self.n = 1 + 2 * len(self.labels)                       # O, B-x, I-x ...
        self.names = ["O"] + [f"{p}-{l}" for l in self.labels for p in ("B", "I")]

    def b(self, label): return 1 + 2 * self.k[label]
    def i(self, label): return 2 + 2 * self.k[label]


def spans_from_ids(lab):
    """Lenient BIO -> [(first_tok, last_tok, label_idx)] over a sequence of REAL tokens."""
    out, cur = [], None
    for t, y in enumerate(lab):
        if y == 0:
            if cur: out.append(tuple(cur)); cur = None
        else:
            k, is_b = (y - 1) // 2, (y - 1) % 2 == 0
            if is_b or cur is None or cur[2] != k:
                if cur: out.append(tuple(cur))
                cur = [t, t, k]
            else:
                cur[1] = t
    if cur: out.append(tuple(cur))
    return out


def parse_gold(raw, text):
    """Accepts [[s,e,label(,txt)], ...] or [{'start','end','label','text'}, ...]. Returns [(s,e,label,txt|None)]."""
    out = []
    for ent in raw or []:
        if isinstance(ent, dict):
            g = lambda *ks: next((ent[k] for k in ks if k in ent), None)
            s, e, l, t = g("start", "start_char", "begin"), g("end", "end_char"), g("label", "type", "entity"), g("text", "entity_text", "span")
        else:
            s, e, l = ent[0], ent[1], ent[2]
            t = ent[3] if len(ent) > 3 else None
        out.append((int(s), int(e), l, t))
    return out


def detect_end_shift(texts, golds, n=3000):
    """0 if gold end is exclusive (text[s:e]==entity_text), 1 if inclusive. Fails loudly if neither matches."""
    ex = inc = tot = 0
    for did, ents in golds.items():
        for s, e, l, t in ents:
            if t is None: continue
            tot += 1
            ex += texts[did][s:e] == t
            inc += texts[did][s:e + 1] == t
            if tot >= n: break
        if tot >= n: break
    if tot == 0:
        print("  [offsets] gold has no entity_text field -> assuming end is EXCLUSIVE (python slice)."); return 0
    print(f"  [offsets] text[s:e]==entity_text for {ex}/{tot};  text[s:e+1]==entity_text for {inc}/{tot}")
    if ex / tot > 0.9: return 0
    if inc / tot > 0.9: return 1
    raise RuntimeError("Gold offsets do not reproduce entity_text under either convention. Inspect a few examples "
                       "(maybe the text was normalised?) before training.")


def assign_bio(cs, ce, ents, ls):
    """Token-level labels (len == #tokens incl. </s>, IGN on </s>). Returns (labels, n_overlap, n_unaligned)."""
    real = np.where(cs >= 0)[0]
    rs, re_ = cs[real], ce[real]
    y = np.full(len(cs), IGN, np.int16)
    lab_real = np.zeros(len(real), np.int16)
    n_over = n_unal = 0
    for s, e, l, _ in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = int(np.searchsorted(rs, s, "left"))
        hi = int(np.searchsorted(re_, e, "right"))
        if hi <= lo: n_unal += 1; continue
        if (lo > 0 and re_[lo - 1] > s) or (hi < len(rs) and rs[hi] < e) or rs[lo] != s or re_[hi - 1] != e:
            n_unal += 1          # entity edge falls inside a token -> cannot be reproduced exactly
        if (lab_real[lo:hi] != 0).any():
            n_over += 1; continue
        lab_real[lo] = ls.b(l); lab_real[lo + 1:hi] = ls.i(l)
    y[real] = lab_real
    return y, n_over, n_unal


def make_windows(n, W, stride):
    if n <= W: return [(0, n)]
    out, a = [], 0
    while True:
        b = min(a + W, n)
        out.append((a, b))
        if b == n: break
        a += stride
    return out


def to_char_spans(lab_real, cs_real, ce_real, ls):
    return [(int(cs_real[i]), int(ce_real[j]), ls.labels[k]) for i, j, k in spans_from_ids(lab_real)]


def strict_scores(truth, pred, labels):
    st = {l: [0, 0, 0] for l in labels}
    for did in truth:
        g = {tuple(x[:3]) for x in truth[did]}; p = {tuple(x[:3]) for x in pred.get(did, [])}
        for l in labels:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            st[l][0] += len(gs & ps); st[l][1] += len(ps - gs); st[l][2] += len(gs - ps)
    per, P, R, F1 = {}, [], [], []
    for l in labels:
        tp, fp, fn = st[l]
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = dict(P=pr, R=rc, F1=f, tp=tp, fp=fp, fn=fn); P.append(pr); R.append(rc); F1.append(f)
    return dict(macro_F1=sum(F1) / len(F1), macro_P=sum(P) / len(P), macro_R=sum(R) / len(R), per_label=per)


def error_breakdown(truth, pred, labels):
    """What kind of mistakes? exact / boundary (same label, overlapping) / label confusion (same span) / spurious / missed."""
    c = dict(exact=0, boundary=0, label_confusion=0, spurious=0, missed=0)
    conf = {}
    for did in truth:
        g = [tuple(x[:3]) for x in truth[did]]; p = [tuple(x[:3]) for x in pred.get(did, [])]
        gset, pset = set(g), set(p)
        for (s, e, l) in p:
            if (s, e, l) in gset: c["exact"] += 1; continue
            same_span = [x for x in g if x[0] == s and x[1] == e]
            ovl_same = [x for x in g if x[2] == l and x[0] < e and s < x[1]]
            ovl_any = [x for x in g if x[0] < e and s < x[1]]
            if same_span: c["label_confusion"] += 1; conf[(same_span[0][2], l)] = conf.get((same_span[0][2], l), 0) + 1
            elif ovl_same: c["boundary"] += 1
            elif ovl_any: c["label_confusion"] += 1
            else: c["spurious"] += 1
        for (s, e, l) in g:
            if (s, e, l) not in pset and not any(x[0] < e and s < x[1] for x in p): c["missed"] += 1
    top = sorted(conf.items(), key=lambda kv: -kv[1])[:8]
    c["top_label_confusions(gold->pred)"] = [[a, b, n] for (a, b), n in top]
    return c


class NERModel(nn.Module):
    def __init__(self, lm, num_labels, causal=True, head_dropout=0.1, head="linear"):
        super().__init__()
        self.token_emb, self.blocks, self.final_norm = lm.token_emb, lm.blocks, lm.final_norm
        D = self.token_emb.embedding_dim
        self.drop = nn.Dropout(head_dropout)
        self.head = nn.Linear(D, num_labels) if head == "linear" else nn.Sequential(
            nn.Linear(D, D), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(D, num_labels))
        self.set_causal(causal)

    def set_causal(self, causal):
        self.causal = causal
        for b in self.blocks: b.attn.causal = causal

    def forward(self, idx, pad_mask=None):
        x = self.token_emb(idx)
        for b in self.blocks: x = b(x, pad_mask)
        return self.head(self.drop(self.final_norm(x)))


class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) B A x   (Hu et al. 2021)"""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMAPlusLinear(nn.Module):
    """LoRMA_+ (Bihany et al. 2025, arXiv 2506.07621): W' = ((alpha/r) B A + I) W0  -- MULTIPLICATIVE on the output side.
    Right-to-left evaluation:  h = W0 x ;  y = h + (alpha/r) B (A h).   B=0, A random -> identity at init."""
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))

    def forward(self, x):
        h = self.base(x)
        return h + self.s * F.linear(F.linear(h, self.A), self.B)


class LoRMAPiLinear(nn.Module):
    """LoRMA_pi: W' = I_pi(B A) W0 where I_pi cyclically rotates row i of BA by i.  Init: B[:,0]=1, A[0,0]=1, rest of A=0
    -> rot(BA)=I.  (Scale of the 'random' part of B is NOT specified in what I read; std=0.02 is an assumption.)"""
    def __init__(self, base, r):
        super().__init__()
        d = base.out_features
        self.base = base
        B = torch.randn(d, r) * 0.02; B[:, 0] = 1.0
        A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        ar = torch.arange(d)
        self.register_buffer("idx", (ar[None, :] - ar[:, None]) % d, persistent=False)   # M[i,j] = P[i,(j-i) mod d]

    def forward(self, x):
        h = self.base(x)
        M = torch.gather(self.B @ self.A, 1, self.idx)
        return F.linear(h, M.to(h.dtype))


class AdapterWrap(nn.Module):
    """Houlsby-style bottleneck adapter applied to the sub-layer output (before the residual add)."""
    def __init__(self, inner, d, bottleneck):
        super().__init__()
        self.inner, self.down, self.up = inner, nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        y = self.inner(*a, **k)
        return y + self.up(F.gelu(self.down(y)))


class IA3Out(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x): return self.base(x) * self.l


class IA3In(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x): return self.base(x * self.l)


ATTN_T = ("q_proj", "k_proj", "v_proj", "out_proj")
FFN_T = ("gate", "up", "down")


def apply_method(model, args):
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for p in model.head.parameters(): p.requires_grad = True                      # the new head is always trained
    D = model.token_emb.embedding_dim
    tg = [t.strip() for t in args.targets.split(",")]

    def wrap_linears(factory, names):
        for b in model.blocks:
            for t in names:
                parent = b.attn if t in ATTN_T else b.ffn
                setattr(parent, t, factory(getattr(parent, t)))

    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "partial":                                                          # layer-wise: top-k blocks + final norm
        for b in model.blocks[-args.top_layers:]:
            for p in b.parameters(): p.requires_grad = True
        for p in model.final_norm.parameters(): p.requires_grad = True
    elif m == "bitfit":                                                           # model has no biases -> tune RMSNorm gains
        for mod in model.modules():
            if isinstance(mod, RMSNorm):
                for p in mod.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: LoRALinear(l, args.rank, args.alpha, args.lora_dropout), tg)
    elif m == "lorma":
        wrap_linears(lambda l: LoRMAPlusLinear(l, args.rank, args.alpha), tg)
    elif m == "lorma_pi":
        wrap_linears(lambda l: LoRMAPiLinear(l, args.rank), [t for t in tg if t in ATTN_T])   # d_out<=768 only (cheap)
    elif m == "adapter":
        for b in model.blocks:
            b.attn = AdapterWrap(b.attn, D, args.adapter_dim)
            b.ffn = AdapterWrap(b.ffn, D, args.adapter_dim)
    elif m == "ia3":
        for b in model.blocks:
            b.attn.k_proj, b.attn.v_proj = IA3Out(b.attn.k_proj), IA3Out(b.attn.v_proj)
            b.ffn.down = IA3In(b.ffn.down)
    else:
        raise ValueError(m)
    return model


def mark_new_params_trainable(model, before_ids):
    """Parameters created by a wrapper (LoRA/LoRMA/adapter/IA3) are trainable; wrapped base weights keep their identity."""
    for p in model.parameters():
        if id(p) not in before_ids: p.requires_grad = True


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


def load_split_files(data_dir):
    def rd(name):
        with open(os.path.join(data_dir, name), encoding="utf-8") as f: return json.load(f)
    with open(os.path.join(data_dir, "labels.txt"), encoding="utf-8") as f:
        labels = [x.strip() for x in f if x.strip()]
    texts, golds = {}, {}
    for fold in ("fold1", "fold2"):
        t, g = rd(f"{fold}-text.json"), rd(f"{fold}-gold.json")
        if isinstance(g, list):                                   # tolerate [{"id":..,"entities":[..]}, ...]
            g = {str(x.get("id")): x.get("entities", x.get("annotations", x.get("labels", []))) for x in g}
        g = {str(k): v for k, v in g.items()}
        for did, txt in t.items():
            did = str(did)
            texts[did] = txt
            golds[did] = parse_gold(g.get(did), txt)
    test = {str(k): v for k, v in rd("fold3-text.json").items()}
    return labels, texts, golds, test


def build_docs(texts, golds, ls, piecer, shift, tag=""):
    docs = []
    n_ent = n_over = n_unal = 0
    for did, txt in texts.items():
        ids, cs, ce = piecer.encode_doc(txt)
        ents = [(s, e + shift, l, t) for s, e, l, t in golds.get(did, [])]
        bad = [x for x in ents if x[2] not in ls.k]
        assert not bad, f"gold label not in labels.txt: {bad[:3]}"
        y, no, nu = assign_bio(cs, ce, ents, ls)
        n_ent += len(ents); n_over += no; n_unal += nu
        docs.append(dict(id=did, ids=ids, cs=cs, ce=ce, y=y, ents=[(s, e, l) for s, e, l, _ in ents]))
    print(f"  [{tag}] docs={len(docs)} tokens={sum(len(d['ids']) for d in docs):,} entities={n_ent:,} "
          f"overlap-dropped={n_over} edge-inside-token={n_unal} ({100 * n_unal / max(1, n_ent):.2f}%)")
    return docs


def oracle_f1(docs, ls):
    """Decode the GOLD token labels back to char spans: this is the ceiling of the whole tokenisation/alignment pipeline."""
    truth, pred = {}, {}
    for d in docs:
        real = d["cs"] >= 0
        pred[d["id"]] = to_char_spans(d["y"][real], d["cs"][real], d["ce"][real], ls)
        truth[d["id"]] = list(set(d["ents"]))
    return strict_scores(truth, pred, ls.labels)["macro_F1"]


def collate(items, device):
    L = max(len(x[0]) for x in items)
    ids = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    lab = torch.full((len(items), L), IGN, dtype=torch.long)
    msk = torch.zeros((len(items), L), dtype=torch.bool)
    for i, (a, b) in enumerate(items):
        ids[i, :len(a)] = torch.from_numpy(a.astype(np.int64)); msk[i, :len(a)] = True
        if b is not None: lab[i, :len(b)] = torch.from_numpy(b.astype(np.int64))
    return ids.to(device), lab.to(device), (None if msk.all() else msk.to(device))


def train_windows(docs, W, stride, neg_keep, rng):
    out = []
    for di, d in enumerate(docs):
        for a, b in make_windows(len(d["ids"]), W, stride):
            has = (d["y"][a:b] > 0).any()
            if has or rng.random() < neg_keep: out.append((di, a, b))
    return out


@torch.no_grad()
def doc_logprobs(model, doc, W, stride, device, bs=16, amp=True):
    n = len(doc["ids"])
    if n == 0: return np.zeros((0, model_nl(model)), np.float32)
    acc = np.zeros((n, model_nl(model)), np.float32); wsum = np.zeros(n, np.float32)
    wins = make_windows(n, W, stride)
    causal = model.causal
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        ids, _, msk = collate([(doc["ids"][a:b], None) for a, b in chunk], device)
        with torch.autocast("cuda", enabled=amp and device.type == "cuda"):
            lg = model(ids, msk)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for j, (a, b) in enumerate(chunk):
            L = b - a; pos = np.arange(L, dtype=np.float32)
            w = pos + 1 if causal else np.minimum(pos + 1, L - pos)
            acc[a:b] += lp[j, :L] * w[:, None]; wsum[a:b] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]


def model_nl(model):
    h = model.head
    return h.out_features if isinstance(h, nn.Linear) else h[-1].out_features


def viterbi(lp, ls):
    T, N = lp.shape
    trans = np.zeros((N, N), np.float32)
    for j in range(1, N):
        if (j - 1) % 2 == 1:                                    # I-x only after B-x / I-x of the same type
            k = (j - 1) // 2
            trans[:, j] = -1e4; trans[1 + 2 * k, j] = 0; trans[2 + 2 * k, j] = 0
    start = np.zeros(N, np.float32); start[[j for j in range(1, N) if (j - 1) % 2 == 1]] = -1e4
    score = lp[0] + start
    bp = np.zeros((T, N), np.int16)
    for t in range(1, T):
        cand = score[:, None] + trans
        bp[t] = cand.argmax(0); score = cand.max(0) + lp[t]
    path = np.zeros(T, np.int64); path[-1] = score.argmax()
    for t in range(T - 1, 0, -1): path[t - 1] = bp[t, path[t]]
    return path


def predict_docs(model, docs, ls, W, stride, device, decodes=("viterbi",), shift=0, amp=True):
    model.eval()
    out = {d: {} for d in decodes}; n_tok = 0; t0 = time.time()
    for doc in docs:
        lp = doc_logprobs(model, doc, W, stride, device, amp=amp)
        real = doc["cs"] >= 0
        lpr, csr, cer = lp[real], doc["cs"][real], doc["ce"][real]
        n_tok += len(doc["ids"])
        for dm in decodes:
            if len(lpr) == 0: out[dm][doc["id"]] = []; continue
            lab = viterbi(lpr, ls) if dm == "viterbi" else lpr.argmax(-1)
            out[dm][doc["id"]] = to_char_spans(lab, csr, cer, ls)
    return out, n_tok / max(1e-6, time.time() - t0)


@torch.no_grad()
def collect_logp(model, docs, W, stride, device, amp=True):
    """Window-merged token log-probs (REAL tokens only, float16) per doc -> saved for ensembling / post-processing."""
    model.eval(); out = {}; n_tok = 0; t0 = time.time()
    for doc in docs:
        lp = doc_logprobs(model, doc, W, stride, device, amp=amp)
        out[doc["id"]] = lp[doc["cs"] >= 0].astype(np.float16); n_tok += len(doc["ids"])
    return out, n_tok / max(1e-6, time.time() - t0)


def decode_logp(lp_dict, docs, ls, mode):
    out = {}
    for d in docs:
        lpr = lp_dict[d["id"]].astype(np.float32); real = d["cs"] >= 0
        if len(lpr) == 0: out[d["id"]] = []; continue
        lab = viterbi(lpr, ls) if mode == "viterbi" else lpr.argmax(-1)
        out[d["id"]] = to_char_spans(lab, d["cs"][real], d["ce"][real], ls)
    return out


def dev_truth_of(docs):
    return {d["id"]: sorted(set(d["ents"])) for d in docs}


def shifted(preds, shift):
    return {k: [[s, e - shift, l] for s, e, l in v] for k, v in preds.items()}


def lr_at(step, total, warm, base):
    if step < warm: return base * (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return base * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


def save_ckpt(path, model, meta):
    torch.save({"state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "meta": meta}, path)


def load_ner_ckpt(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck["meta"]
    lm, _ = build_lm(meta["lm_cfg"])
    ls = LabelSpace(meta["labels"])
    model = NERModel(lm, ls.n, causal=meta["causal"], head=meta.get("head", "linear"))
    assert meta["method"] in ("full", "kd", "partial", "bitfit"), "teacher must be a method without structural wrappers"
    model.load_state_dict(ck["state"]); return model.to(device).eval(), meta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True); ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--init", required=True, help="Phase-1 weights (model_step45000.pt / model.pt / checkpoint.pt)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--method", required=True, choices=["full", "partial", "bitfit", "lora", "lorma", "lorma_pi", "adapter", "ia3", "kd"])
    ap.add_argument("--attn", default="bidir", choices=["causal", "bidir"])
    ap.add_argument("--head", default="linear", choices=["linear", "mlp"])
    ap.add_argument("--epochs", type=float, default=3.0); ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=None); ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.01); ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--window", type=int, default=512); ap.add_argument("--stride", type=int, default=384)
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--targets", default="q_proj,k_proj,v_proj,out_proj,gate,up,down")
    ap.add_argument("--adapter_dim", type=int, default=64); ap.add_argument("--top_layers", type=int, default=4)
    ap.add_argument("--o_weight", type=float, default=0.5, help="loss weight of the 'O' class (class balancing)")
    ap.add_argument("--neg_keep", type=float, default=0.5, help="fraction of entity-free windows kept per epoch")
    ap.add_argument("--dev_frac", type=float, default=0.06); ap.add_argument("--dev_docs", type=int, default=200)
    ap.add_argument("--full_data", action="store_true", help="also train on the dev docs (use for final runs only)")
    ap.add_argument("--time_budget_min", type=float, default=90.0)
    ap.add_argument("--teacher", default=None); ap.add_argument("--student_layers", type=int, default=6)
    ap.add_argument("--kd_T", type=float, default=2.0); ap.add_argument("--kd_alpha", type=float, default=0.5)
    ap.add_argument("--max_train_docs", type=int, default=0); ap.add_argument("--no_test", action="store_true")
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--cache", default=None)
    ap.add_argument("--n_folds", type=int, default=0, help="k-fold CV over the 70 labelled docs (0 = old single dev split)")
    ap.add_argument("--fold", type=int, default=0, help="which fold is the dev fold when --n_folds > 0")
    args = ap.parse_args()

    t_global = time.time()
    os.makedirs(args.out, exist_ok=True)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    default_lr = dict(full=1e-4, kd=1e-4, partial=1e-4, bitfit=2e-3, lora=1e-3, lorma=1e-3, lorma_pi=5e-4, adapter=1e-3, ia3=3e-3)
    base_lr = args.lr if args.lr is not None else default_lr[args.method]
    print(f"== method={args.method} attn={args.attn} lr={base_lr} device={device}")

    tok = ByteLevelBPETokenizer.load(args.tokenizer)
    labels, texts, golds, test_texts = load_split_files(args.data_dir)
    assert len(labels) == 12, labels
    ls = LabelSpace(labels)
    shift = detect_end_shift(texts, golds)
    key = hashlib.md5((open(args.tokenizer, "rb").read()[:100000] + str(shift).encode() + str(len(texts)).encode())).hexdigest()[:10]
    cache = args.cache or os.path.join(args.data_dir, f"tokcache_{key}.pkl")
    piecer = Piecer(tok)
    if os.path.exists(cache):
        with open(cache, "rb") as f: all_docs, test_docs = pickle.load(f)
        print(f"  loaded tokenised cache {cache}")
    else:
        all_docs = build_docs(texts, golds, ls, piecer, shift, "fold1+2")
        test_docs = [dict(id=k, ids=a, cs=b, ce=c) for k, v in test_texts.items() for a, b, c in [piecer.encode_doc(v)]]
        tmp = f"{cache}.{os.getpid()}.tmp"
        with open(tmp, "wb") as f: pickle.dump((all_docs, test_docs), f)
        os.replace(tmp, cache)
    order = sorted(d["id"] for d in all_docs)
    random.Random(1234).shuffle(order)                                    # fixed split shared by ALL runs
    if args.n_folds > 0:                                                  # k-fold CV: dev = fold `fold`, train = the other folds
        assert 0 <= args.fold < args.n_folds
        dev_ids = set(np.array_split(np.array(order), args.n_folds)[args.fold].tolist())
        dev_docs = [d for d in all_docs if d["id"] in dev_ids]
    else:
        n_dev = max(1, int(len(order) * args.dev_frac))
        dev_ids = set(order[:n_dev])
        dev_docs = [d for d in all_docs if d["id"] in dev_ids][:args.dev_docs]
    train_docs = [d for d in all_docs if (d["id"] not in dev_ids or args.full_data)]
    if args.max_train_docs: train_docs = train_docs[:args.max_train_docs]
    print(f"  train docs={len(train_docs)} dev docs={len(dev_docs)} test docs={len(test_docs)}")
    print(f"  ORACLE strict macro-F1 of the tokenisation/alignment pipeline: train={oracle_f1(train_docs, ls):.4f} dev={oracle_f1(dev_docs, ls):.4f}")
    cnt = np.zeros(ls.n, np.int64)
    for d in train_docs: cnt += np.bincount(d["y"][d["y"] >= 0].astype(np.int64), minlength=ls.n)
    print("  token label counts:", {ls.names[i]: int(c) for i, c in enumerate(cnt) if i == 0 or i % 2 == 1})

    teacher = None
    if args.method == "kd":
        assert args.teacher, "--teacher runs/<full_run>/best.pt is required for kd"
        teacher, tmeta = load_ner_ckpt(args.teacher, device)
        layer_map = [int(round(x)) for x in np.linspace(0, tmeta["lm_cfg"]["n_layers"] - 1, args.student_layers)]
        print(f"  KD student = teacher blocks {layer_map}; teacher params={count_params(teacher)[0] / 1e6:.1f}M")
        lm, lm_cfg = load_phase1(args.init, layer_map=layer_map)
        args.attn = "causal" if tmeta["causal"] else "bidir"     # student sees the same context as the teacher
        args.head = tmeta.get("head", "linear")                  # so the teacher head can initialise the student head
    else:
        lm, lm_cfg = load_phase1(args.init)
    assert args.window <= lm_cfg["max_seq_len"], "window longer than the Phase-1 context (RoPE tables are built for 512)"
    model = NERModel(lm, ls.n, causal=(args.attn == "causal"), head=args.head)
    if args.method == "kd":
        with torch.no_grad():                                             # start student head from the teacher head
            model.head.load_state_dict(teacher.head.state_dict())
    before = {id(p) for p in model.parameters()}
    apply_method(model, args)
    mark_new_params_trainable(model, before)
    model.to(device)
    tot, trn = count_params(model)
    print(f"  params total={tot / 1e6:.2f}M trainable={trn / 1e6:.3f}M ({100 * trn / tot:.2f}%)")

    decay, no_decay, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (head_p if n.startswith("head.") else decay if p.ndim >= 2 else no_decay).append(p)
    groups = [dict(params=head_p, lr=args.lr_head, weight_decay=0.0), dict(params=no_decay, lr=base_lr, weight_decay=0.0),
              dict(params=decay, lr=base_lr, weight_decay=args.wd)]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    cw = torch.ones(ls.n, device=device); cw[0] = args.o_weight

    rng = random.Random(args.seed)
    meta = dict(method=args.method, causal=model.causal, labels=labels, lm_cfg=lm_cfg, head=args.head)
    best_path = os.path.join(args.out, "best.pt")
    wins = train_windows(train_docs, args.window, args.stride, args.neg_keep, rng)
    steps_per_epoch = max(1, len(wins) // args.bs)
    planned = int(args.epochs * steps_per_epoch)
    total, step, best_f1, best_step, t_bench = planned, 0, -1.0, 0, None
    eval_every = None
    print(f"  windows/epoch={len(wins)} steps/epoch={steps_per_epoch} planned steps={planned}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()

    def run_eval(tag):
        nonlocal best_f1, best_step
        pr, _ = predict_docs(model, dev_docs, ls, args.window, args.stride, device, ("viterbi",), amp=amp)
        sc = strict_scores(dev_truth_of(dev_docs), pr["viterbi"], labels)
        print(f"  [eval {tag}] step={step} dev strict macro-F1={sc['macro_F1']:.4f} P={sc['macro_P']:.4f} R={sc['macro_R']:.4f}", flush=True)
        if sc["macro_F1"] > best_f1:
            best_f1, best_step = sc["macro_F1"], step; save_ckpt(best_path, model, meta)
        model.train()

    model.train()
    t_start = time.time(); done = False; ep = 0; loss_ema = None
    while not done:
        order_w = train_windows(train_docs, args.window, args.stride, args.neg_keep, rng); rng.shuffle(order_w); ep += 1   # fresh negative-window sample each epoch
        for bi in range(0, len(order_w) - args.bs + 1, args.bs):
            if step == 5: t_bench = time.time()
            if step == 35 and t_bench is not None:                      # fix the schedule from the measured speed
                sps = (time.time() - t_bench) / 30
                left = args.time_budget_min * 60 * 0.80 - (time.time() - t_global)
                total = max(60, min(planned, 35 + int(left / sps)))
                eval_every = max(50, steps_per_epoch)                     # evaluate (and keep the best) once per epoch
                print(f"  speed={1 / sps:.2f} steps/s -> total steps={total} (planned {planned}); eval every {eval_every}", flush=True)
            if step >= total or (time.time() - t_global) > args.time_budget_min * 60 * 0.95: done = True; break
            batch = [(train_docs[di], a, b) for di, a, b in order_w[bi:bi + args.bs]]
            ids, lab, msk = collate([(d["ids"][a:b], d["y"][a:b]) for d, a, b in batch], device)
            lr_scale = lr_at(step, total, args.warmup, 1.0)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_scale
            with torch.autocast("cuda", enabled=amp):
                logits = model(ids, msk)
            lf = logits.float()
            ce = F.cross_entropy(lf.view(-1, ls.n), lab.view(-1), weight=cw, ignore_index=IGN)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", enabled=amp):
                    tl = teacher(ids, msk).float()
                m = lab.view(-1) != IGN
                T = args.kd_T
                kd = F.kl_div(F.log_softmax(lf.view(-1, ls.n)[m] / T, -1), F.softmax(tl.view(-1, ls.n)[m] / T, -1),
                              reduction="batchmean") * T * T
                loss = args.kd_alpha * ce + (1 - args.kd_alpha) * kd
            else:
                loss = ce
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            loss_ema = loss.item() if loss_ema is None else 0.98 * loss_ema + 0.02 * loss.item()
            if step % 50 == 0:
                print(f"  step {step}/{total} ep {ep} loss {loss_ema:.4f} lr {base_lr * lr_scale:.2e} "
                      f"{(time.time() - t_start) / 60:.1f}min", flush=True)
            if eval_every and step % eval_every == 0 and step < total: run_eval("mid")
    train_min = (time.time() - t_start) / 60
    run_eval("final")
    peak_gb = torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else 0.0

    ck = torch.load(best_path, map_location="cpu", weights_only=False); model.load_state_dict(ck["state"]); model.to(device)
    model.eval()
    dev_lp, tps = collect_logp(model, dev_docs, args.window, args.stride, device, amp)
    np.savez_compressed(os.path.join(args.out, "dev_logp.npz"), **dev_lp)
    pr = {dm: decode_logp(dev_lp, dev_docs, ls, dm) for dm in ("greedy", "viterbi")}
    truth = dev_truth_of(dev_docs)
    sc = {dm: strict_scores(truth, pr[dm], labels) for dm in pr}
    use = max(sc, key=lambda k: sc[k]["macro_F1"])
    print(f"  dev macro-F1: greedy={sc['greedy']['macro_F1']:.4f} viterbi={sc['viterbi']['macro_F1']:.4f} -> use {use}")
    for l in labels:
        r = sc[use]["per_label"][l]; print(f"    {l:10s} P={r['P']:.3f} R={r['R']:.3f} F1={r['F1']:.3f}  (tp={r['tp']} fp={r['fp']} fn={r['fn']})")
    with open(os.path.join(args.out, "dev_truth.json"), "w") as f:
        json.dump({k: [[s, e - shift, l] for s, e, l in v] for k, v in truth.items()}, f)
    with open(os.path.join(args.out, "dev_pred.json"), "w") as f: json.dump(shifted(pr[use], shift), f)

    res = dict(method=args.method, attn=args.attn, params_total=tot, params_trainable=trn, train_minutes=train_min,
               steps=step, planned_steps=planned, peak_gpu_gb=peak_gb, infer_tokens_per_s=tps, best_step=best_step,
               dev_macro_F1=sc[use]["macro_F1"], dev_macro_P=sc[use]["macro_P"], dev_macro_R=sc[use]["macro_R"],
               dev_greedy_F1=sc["greedy"]["macro_F1"], dev_viterbi_F1=sc["viterbi"]["macro_F1"], decode=use,
               per_label=sc[use]["per_label"], errors=error_breakdown(truth, pr[use], labels), args=vars(args),
               shift=shift, fold=args.fold if args.n_folds > 0 else None, n_folds=args.n_folds, dev_ids=sorted(dev_ids), n_dev_docs=len(dev_docs))
    if not args.no_test:
        test_lp, _ = collect_logp(model, test_docs, args.window, args.stride, device, amp)
        np.savez_compressed(os.path.join(args.out, "test_logp.npz"), **test_lp)
        pred = shifted(decode_logp(test_lp, test_docs, ls, use), shift)
        assert set(pred) == {d["id"] for d in test_docs}
        pj = os.path.join(args.out, "prediction.json")
        with open(pj, "w") as f: json.dump(pred, f)
        with zipfile.ZipFile(os.path.join(args.out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
        res["n_test_docs"], res["n_test_entities"] = len(pred), sum(len(v) for v in pred.values())
        print(f"  wrote {args.out}/submission.zip  ({res['n_test_entities']} entities in {len(pred)} docs)")
    with open(os.path.join(args.out, "results.json"), "w") as f: json.dump(res, f, indent=1, default=str)
    print(f"== done in {(time.time() - t_global) / 60:.1f} min | dev strict macro-F1 = {res['dev_macro_F1']:.4f}")


if __name__ == "__main__":
    main()

Writing lner.py


In [2]:
%%writefile lner_ret.py
#!/usr/bin/env python3
"""
lner_ret.py -- CS781 Phase 2 (Legal NER) on top of YOUR Phase-1 *retention-lite* model (RetNet-style, 73.6M params).

Thin layer over lner.py (keep both files in the same folder): it swaps in the retention model + its PEFT wiring and then
calls lner.main(), so tokenisation, BIO labelling, windowing, 5-fold CV, Viterbi, strict scoring, log-prob saving and
ensemble.py all behave EXACTLY as for the decoder.

The 4 methods offered here:  full | lora | lorma | kd
  python lner_ret.py --data_dir p2data --tokenizer bpe_tokenizer.json --init model.pt --out runs/ret_lora --method lora --window 1024 --stride 768
The retention model is causal-only (a recurrence), so there is no bidirectional option.

Sections: A. retention model (copied from your notebook, logic unchanged)  B. loader / wrapper / PEFT wiring  C. entry point
"""
import math
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import lner

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos[:, :, offset:offset + T, :].to(x.dtype)
        sin = self.sin[:, :, offset:offset + T, :].to(x.dtype)
        return x * cos + rotate_half(x) * sin


class GeGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.gelu(self.gate(x)) * self.up(x)))


def retention_chunkwise(q, k, v, log_gamma, state=None, chunk_size=128):
    B, H, T, dk = q.shape
    dv = v.size(-1)
    if state is None:
        state = q.new_zeros(B, H, dk, dv)
    lg = log_gamma.view(1, H, 1, 1)
    outs = []
    for s in range(0, T, chunk_size):
        qc, kc, vc = q[:, :, s:s + chunk_size], k[:, :, s:s + chunk_size], v[:, :, s:s + chunk_size]
        L = qc.size(2)
        idx = torch.arange(L, device=q.device, dtype=q.dtype)
        diff = idx[:, None] - idx[None, :]
        D = torch.exp(diff.clamp(min=0)[None, None] * lg) * (diff >= 0)
        inner = ((qc @ kc.transpose(-1, -2)) * D) @ vc
        q_decay = torch.exp((idx + 1).view(1, 1, L, 1) * lg)
        cross = (qc * q_decay) @ state
        outs.append(inner + cross)
        k_decay = torch.exp((L - 1 - idx).view(1, 1, L, 1) * lg)
        state = torch.exp(L * lg) * state + (kc * k_decay).transpose(-1, -2) @ vc
    return torch.cat(outs, dim=2), state


class MultiScaleRetention(nn.Module):
    def __init__(self, d_model, n_heads, rope, chunk_size, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head, self.chunk_size = n_heads, d_model // n_heads, chunk_size
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.g_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.head_norm_w = nn.Parameter(torch.ones(n_heads, 1, self.d_head))
        self.rope = rope
        self.dropout = nn.Dropout(dropout)
        gamma = 1.0 - 2.0 ** (-5.0 - torch.arange(n_heads, dtype=torch.float32))
        self.register_buffer("log_gamma", torch.log(gamma), persistent=False)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x, state=None, offset=0):
        B, T, C = x.shape
        q = self._split(self.q_proj(x)) * (self.d_head ** -0.5)
        k = self._split(self.k_proj(x))
        v = self._split(self.v_proj(x))
        q, k = self.rope(q, offset), self.rope(k, offset)
        with torch.autocast(device_type=x.device.type, enabled=False):
            o, new_state = retention_chunkwise(q.float(), k.float(), v.float(), self.log_gamma, state, self.chunk_size)
            o = o * torch.rsqrt(o.pow(2).mean(dim=-1, keepdim=True) + 1e-6) * self.head_norm_w
        o = o.to(x.dtype).transpose(1, 2).contiguous().view(B, T, C)
        o = o * F.silu(self.g_proj(x))
        return self.dropout(self.out_proj(o)), new_state


class RetentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, rope, chunk_size, dropout):
        super().__init__()
        self.norm_ret = RMSNorm(d_model)
        self.retention = MultiScaleRetention(d_model, n_heads, rope, chunk_size, dropout)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = GeGLU(d_model, d_ff, dropout)

    def forward(self, x, state=None, offset=0):
        h, new_state = self.retention(self.norm_ret(x), state=state, offset=offset)
        x = x + h
        return x + self.ffn(self.norm_ffn(x)), new_state


class RetentionLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_pos,
                 chunk_size=128, dropout=0.1, rope_base=10000.0, tie_weights=True):
        super().__init__()
        self.max_pos = max_pos
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_heads, max_pos, rope_base)
        self.blocks = nn.ModuleList([RetentionBlock(d_model, n_heads, d_ff, self.rope, chunk_size, dropout) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_emb.weight

    def _run(self, idx, states=None, offset=0):
        if offset + idx.size(1) > self.max_pos:
            raise ValueError(f"Position {offset + idx.size(1)} exceeds RoPE table size {self.max_pos}")
        x = self.emb_dropout(self.token_emb(idx))
        new_states = []
        for i, blk in enumerate(self.blocks):
            x, s = blk(x, state=None if states is None else states[i], offset=offset)
            new_states.append(s)
        return self.lm_head(self.final_norm(x)), new_states


DEFAULT_RET_CFG = dict(vocab_size=12000, d_model=640, n_layers=12, n_heads=10, d_ff=1792, max_pos=4096,
                       chunk_size=128, dropout=0.1, rope_base=10000.0)


def build_lm_ret(cfg):
    c = {**DEFAULT_RET_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_RET_CFG}}
    lm = RetentionLM(**c, tie_weights=True)
    c["max_seq_len"] = c["max_pos"]          # lner.main asserts  window <= lm_cfg["max_seq_len"]
    return lm, c


def load_phase1_ret(path, n_layers=None, layer_map=None):
    """Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config}) or a raw state_dict."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], dict(ck.get("config", {}))
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]; rc = ck.get("run_config", {}) or {}
        cfg = {k.lower(): v for k, v in rc.items()}
        cfg = {**cfg, **{k: v for k, v in dict(vocab_size=rc.get("VOCAB_SIZE"), d_model=rc.get("D_MODEL"), n_heads=rc.get("N_HEADS"),
                                               d_ff=rc.get("D_FF"), max_pos=rc.get("MAX_POS"), chunk_size=rc.get("CHUNK_SIZE")).items() if v is not None}}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    cfg["n_layers"] = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["blocks.0.ffn.gate.weight"].shape[0]
    cfg["n_heads"] = sd["blocks.0.retention.head_norm_w"].shape[0]          # (H, 1, d_head)
    if layer_map is not None:                                                # layer-dropped KD student
        new = {k: v for k, v in sd.items() if not k.startswith("blocks.")}
        for ni, oi in enumerate(layer_map):
            for k, v in sd.items():
                if k.startswith(f"blocks.{oi}."):
                    new[f"blocks.{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_layers"] = new, len(layer_map)
    lm, cfg = build_lm_ret(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k != "lm_head.weight"] + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg


class RetNER(nn.Module):
    """Retention backbone + token-classification head.  Causal only; right padding cannot leak into real tokens."""
    def __init__(self, lm, num_labels, causal=True, head_dropout=0.1, head="linear"):
        super().__init__()
        self.token_emb, self.emb_dropout, self.blocks, self.final_norm = lm.token_emb, lm.emb_dropout, lm.blocks, lm.final_norm
        D = self.token_emb.embedding_dim
        self.drop = nn.Dropout(head_dropout)
        self.head = nn.Linear(D, num_labels) if head == "linear" else nn.Sequential(
            nn.Linear(D, D), nn.GELU(), nn.Dropout(head_dropout), nn.Linear(D, num_labels))
        self.causal = True                                                    # forced: a recurrence has no right context

    def set_causal(self, causal):
        self.causal = True

    def forward(self, idx, pad_mask=None):
        x = self.emb_dropout(self.token_emb(idx))
        for b in self.blocks:
            x, _ = b(x)
        return self.head(self.drop(self.final_norm(x)))


RET_T = ("q_proj", "k_proj", "v_proj", "g_proj", "out_proj")      # inside block.retention
FFN_T = ("gate", "up", "down")                                    # inside block.ffn
_DECODER_DEFAULT_TARGETS = "q_proj,k_proj,v_proj,out_proj,gate,up,down"


def apply_method_ret(model, args):
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for p in model.head.parameters(): p.requires_grad = True
    targets = args.targets
    if targets == _DECODER_DEFAULT_TARGETS: targets += ",g_proj"        # retention has an extra gate projection
    tg = [t.strip() for t in targets.split(",")]

    def wrap_linears(factory):
        for b in model.blocks:
            for t in tg:
                parent = b.retention if t in RET_T else b.ffn
                setattr(parent, t, factory(getattr(parent, t)))

    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: lner.LoRALinear(l, args.rank, args.alpha, args.lora_dropout))
    elif m == "lorma":
        wrap_linears(lambda l: lner.LoRMAPlusLinear(l, args.rank, args.alpha))
    else:
        raise ValueError(f"retention pipeline supports only full | lora | lorma | kd, got {m}")
    return model


def load_ner_ckpt_ret(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck["meta"]
    lm, _ = build_lm_ret(meta["lm_cfg"])
    ls = lner.LabelSpace(meta["labels"])
    model = RetNER(lm, ls.n, causal=True, head=meta.get("head", "linear"))
    assert meta["method"] in ("full", "kd"), "teacher must be a full / kd run"
    model.load_state_dict(ck["state"]); return model.to(device).eval(), meta


def _patched():
    lner.load_phase1, lner.build_lm = load_phase1_ret, build_lm_ret
    lner.NERModel, lner.apply_method, lner.load_ner_ckpt = RetNER, apply_method_ret, load_ner_ckpt_ret


if __name__ == "__main__":
    _patched()
    argv = sys.argv
    if "--window" not in argv: argv += ["--window", "1024"]
    if "--stride" not in argv: argv += ["--stride", "768"]
    if "--attn" not in argv: argv += ["--attn", "causal"]
    sys.argv = argv
    lner.main()

Writing lner_ret.py


In [3]:
%%writefile ensemble.py
#!/usr/bin/env python3
"""
ensemble.py -- combine k-fold runs / several methods, score them OUT-OF-FOLD on all labelled docs, post-process, write a submission.

  python ensemble.py --data_dir p2data --runs_dir runs --names lora,lorma --out ens_lora_lorma --grid
  python ensemble.py --data_dir p2data --runs_dir runs --names lora,lorma,full --out ens_all --min_conf 0.5 --consistency 1

A "name" is a run-family: runs/<name>_f0 ... runs/<name>_f4 (written by lner.py --n_folds 5 --fold k).
Out-of-fold (OOF) score of a family = every labelled doc is predicted by the fold-model that never saw it -> strict macro-F1 on all 70 docs.
Ensembling = average of token probabilities over all selected runs (OOF: over the families for that doc; test: over every run).
"""
import argparse, glob, json, os, pickle, re, zipfile
from collections import Counter, defaultdict
import numpy as np
import lner


def load_runs(runs_dir, name):
    dirs = sorted(glob.glob(os.path.join(runs_dir, f"{name}_f[0-9]*")))
    runs = []
    for d in dirs:
        if all(os.path.exists(os.path.join(d, f)) for f in ("results.json", "dev_logp.npz", "test_logp.npz")):
            runs.append(d)
    return runs


def avg_probs(list_of_lp):
    return np.log(np.maximum(np.mean([np.exp(x.astype(np.float32)) for x in list_of_lp], 0), 1e-9))


def decode_doc(lpr, csr, cer, ls, min_conf):
    path = lner.viterbi(lpr, ls)
    P = np.exp(lpr); out = []
    for i, j, k in lner.spans_from_ids(path):
        conf = float((P[i:j + 1, 1 + 2 * k] + P[i:j + 1, 2 + 2 * k]).mean())      # probability mass on label k (B or I)
        if conf >= min_conf: out.append((int(csr[i]), int(cer[j]), ls.labels[k], conf))
    return out


def consistency(text, spans, allowed, min_count=2, share=0.8, minlen=4):
    """If the same string was tagged with the same label (>=share of >=min_count times) add the untagged occurrences."""
    cnt = defaultdict(Counter)
    for s, e, l, _ in spans: cnt[text[s:e]][l] += 1
    covered = [(s, e) for s, e, _, _ in spans]; new = []
    for string, c in cnt.items():
        tot = sum(c.values()); l, top = c.most_common(1)[0]
        if len(string) < minlen or tot < min_count or top / tot < share or l not in allowed: continue
        for m in re.finditer(r"(?<!\w)" + re.escape(string) + r"(?!\w)", text):
            s, e = m.span()
            if any(s < ce and cs < e for cs, ce in covered): continue
            new.append((s, e, l, 1.0)); covered.append((s, e))
    return spans + new


def postprocess(lp_by_doc, docs, texts, ls, min_conf, cons, cons_labels):
    out = {}
    for d in docs:
        lpr = lp_by_doc[d["id"]].astype(np.float32); real = d["cs"] >= 0
        sp = decode_doc(lpr, d["cs"][real], d["ce"][real], ls, min_conf) if len(lpr) else []
        if cons: sp = consistency(texts[d["id"]], sp, cons_labels)
        out[d["id"]] = [(s, e, l) for s, e, l, _ in sp]
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True); ap.add_argument("--runs_dir", required=True)
    ap.add_argument("--names", required=True, help="comma-separated run families, e.g. lora,lorma,full")
    ap.add_argument("--out", required=True)
    ap.add_argument("--min_conf", type=float, default=0.0); ap.add_argument("--consistency", type=int, default=0)
    ap.add_argument("--cons_labels", default="APP,RESP,COURT,AUTH,JUDGE,WIT,A.COUNSEL,R.COUNSEL,STAT,PREC")
    ap.add_argument("--cache", default=None, help="tokcache_*.pkl to use (default: newest in data_dir)")
    ap.add_argument("--grid", action="store_true", help="print OOF F1 for several post-processing settings, then use the flags above")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    labels, texts, golds, test_texts = lner.load_split_files(args.data_dir)
    ls = lner.LabelSpace(labels)
    cache = args.cache or sorted(glob.glob(os.path.join(args.data_dir, "tokcache_*.pkl")), key=os.path.getmtime)[-1]
    print("  token cache:", cache)
    with open(cache, "rb") as f: all_docs, test_docs = pickle.load(f)
    by_id = {d["id"]: d for d in all_docs}
    names = [n.strip() for n in args.names.split(",")]
    fam = {n: load_runs(args.runs_dir, n) for n in names}
    for n, r in fam.items(): print(f"  family {n}: {len(r)} runs")
    assert all(fam.values()), "a family has no finished runs"
    shift = json.load(open(os.path.join(fam[names[0]][0], "results.json")))["shift"]
    truth_all = {d["id"]: sorted(set(d["ents"])) for d in all_docs}

    oof = {}
    for n, runs in fam.items():
        oof[n] = {}
        for r in runs:
            z = np.load(os.path.join(r, "dev_logp.npz"))
            for k in z.files:
                oof[n][k] = z[k]
                assert len(z[k]) == int((by_id[k]["cs"] >= 0).sum()), (f"{r}: token count differs from the cache ({len(z[k])} vs {int((by_id[k]['cs'] >= 0).sum())}) -- "
                                                          "families trained with DIFFERENT tokenizers cannot be averaged; pass --cache of the matching tokenizer or ensemble them separately")
        print(f"  {n}: OOF coverage {len(oof[n])}/{len(all_docs)} docs")
    common = sorted(set.intersection(*[set(v) for v in oof.values()]))
    docs_c = [by_id[i] for i in common]
    truth = {i: truth_all[i] for i in common}

    print(f"\nOUT-OF-FOLD strict macro-F1 on {len(common)} labelled docs (viterbi, no post-processing)")
    rows = []
    for n in names:
        pr = lner.decode_logp({i: oof[n][i] for i in common}, docs_c, ls, "viterbi")
        sc = lner.strict_scores(truth, pr, labels); rows.append((n, sc))
        print(f"  {n:14s} F1={sc['macro_F1']:.4f}  P={sc['macro_P']:.4f}  R={sc['macro_R']:.4f}")
    ens_oof = {i: avg_probs([oof[n][i] for n in names]) for i in common}
    pr = lner.decode_logp(ens_oof, docs_c, ls, "viterbi")
    sc = lner.strict_scores(truth, pr, labels)
    print(f"  {'ENSEMBLE':14s} F1={sc['macro_F1']:.4f}  P={sc['macro_P']:.4f}  R={sc['macro_R']:.4f}")
    cons_labels = set(x.strip() for x in args.cons_labels.split(","))

    if args.grid:
        print("\nPost-processing grid on the OOF ensemble (min_conf x consistency)  -> macro-F1 / P / R")
        for mc in (0.0, 0.3, 0.4, 0.5, 0.6, 0.7):
            for cn in (0, 1):
                p2 = postprocess(ens_oof, docs_c, texts, ls, mc, cn, cons_labels)
                s2 = lner.strict_scores(truth, p2, labels)
                print(f"  min_conf={mc:.1f} consistency={cn}  F1={s2['macro_F1']:.4f} P={s2['macro_P']:.4f} R={s2['macro_R']:.4f}")
    final_oof = lner.strict_scores(truth, postprocess(ens_oof, docs_c, texts, ls, args.min_conf, args.consistency, cons_labels), labels)
    print(f"\nchosen: min_conf={args.min_conf} consistency={args.consistency} -> OOF macro-F1 {final_oof['macro_F1']:.4f}")
    for l in labels:
        r = final_oof["per_label"][l]; print(f"    {l:10s} P={r['P']:.3f} R={r['R']:.3f} F1={r['F1']:.3f}")

    acc, cnt = {}, 0
    for n, runs in fam.items():
        for r in runs:
            z = np.load(os.path.join(r, "test_logp.npz")); cnt += 1
            for k in z.files:
                acc[k] = acc.get(k, 0) + np.exp(z[k].astype(np.float32))
    test_lp = {k: np.log(np.maximum(v / cnt, 1e-9)) for k, v in acc.items()}
    pred = postprocess(test_lp, test_docs, test_texts, ls, args.min_conf, args.consistency, cons_labels)
    pred = {k: [[s, e - shift, l] for s, e, l in v] for k, v in pred.items()}
    assert set(pred) == {d["id"] for d in test_docs}
    pj = os.path.join(args.out, "prediction.json")
    json.dump(pred, open(pj, "w"))
    with zipfile.ZipFile(os.path.join(args.out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    json.dump(dict(names=names, n_runs=cnt, oof_ensemble_F1=final_oof["macro_F1"], per_family_oof={n: s["macro_F1"] for n, s in rows},
                   min_conf=args.min_conf, consistency=args.consistency, n_test_entities=sum(len(v) for v in pred.values()),
                   per_label=final_oof["per_label"]), open(os.path.join(args.out, "ensemble_report.json"), "w"), indent=1)
    print(f"wrote {args.out}/submission.zip from {cnt} runs ({sum(len(v) for v in pred.values())} entities in {len(pred)} docs)")


if __name__ == "__main__":
    main()

Writing ensemble.py


In [4]:
import os, sys, json, time, glob, subprocess, shutil, hashlib
import numpy as np, pandas as pd
WORK = "/kaggle/working"
def find(name, root="/kaggle/input"):
    return sorted([os.path.join(d, name) for d, _, fs in os.walk(root, followlinks=True) if name in fs], key=os.path.getmtime)
import torch
cands = find("model.pt")
ret = [c for c in cands if "retention" in c.lower()] or cands
INIT = ret[-1] if ret else None
toks = find("bpe_tokenizer.json"); TOK = toks[-1] if toks else None
assert INIT, f"no model.pt found. /kaggle/input has: {os.listdir('/kaggle/input')}"
assert TOK, "no bpe_tokenizer.json found -> attach the dataset that holds the retention model's tokenizer"
print("INIT:", INIT, "\nTOK :", TOK)
for t in toks: print("   tokenizer candidate:", t, hashlib.md5(open(t, "rb").read()).hexdigest()[:8])
sys.path.insert(0, "."); import lner, lner_ret
lm, cfg = lner_ret.load_phase1_ret(INIT)
tok = lner.ByteLevelBPETokenizer.load(TOK)
print("retention cfg:", {k: v for k, v in cfg.items()}, "| params %.1fM" % (sum(p.numel() for p in lm.parameters()) / 1e6))
assert tok.vocab_size == cfg["vocab_size"], f"TOKENIZER MISMATCH: tokenizer vocab {tok.vocab_size} vs model {cfg['vocab_size']} -> wrong bpe_tokenizer.json"
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

INIT: /kaggle/input/notebooks/akshatkasarwal/light-weight-part-2/cs781_phase1/retention_lite_resume20k/model.pt 
TOK : /kaggle/input/notebooks/akshatkasarwal/efficient-lightweight/cs781_phase1/retention_lite_train_supreme_court-train_acts-train_high_courts-train_tribunals/bpe_tokenizer.json
   tokenizer candidate: /kaggle/input/notebooks/akshatkasarwal/efficient-lightweight/cs781_phase1/retention_lite_train_supreme_court-train_acts-train_high_courts-train_tribunals/bpe_tokenizer.json d546b2bf
retention cfg: {'vocab_size': 12000, 'd_model': 640, 'n_layers': 12, 'n_heads': 10, 'd_ff': 1792, 'max_pos': 4096, 'chunk_size': 128, 'dropout': 0.0, 'rope_base': 10000.0, 'max_seq_len': 4096} | params 73.6M
GPUs: 2 ['Tesla T4', 'Tesla T4']


## 1. Data

In [5]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HuggingFace")
DATA = f"{WORK}/p2data_ret"; os.makedirs(DATA, exist_ok=True)
FILES = {"Phase_2_LNER/Train/fold1-text.json":"fold1-text.json","Phase_2_LNER/Train/fold1-gold.json":"fold1-gold.json",
         "Phase_2_LNER/Train/fold2-text.json":"fold2-text.json","Phase_2_LNER/Train/fold2-gold.json":"fold2-gold.json",
         "Phase_2_LNER/Train/labels.txt":"labels.txt","Phase_2_LNER/Test/fold3-text.json":"fold3-text.json"}
for remote, local in FILES.items():
    if not os.path.exists(f"{DATA}/{local}"):
        shutil.copy(hf_hub_download("Exploration-Lab/CS-781-Capstone", remote, repo_type="dataset", token=hf_token), f"{DATA}/{local}")
print(sorted(os.listdir(DATA)))

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

['fold1-gold.json', 'fold1-text.json', 'fold2-gold.json', 'fold2-text.json', 'fold3-text.json', 'labels.txt']


## 2. Sanity checks on GPU: speed / memory of the retention backbone at window 1024, and does the tokenizer behave (no unknown pieces, EOS/PAD ids)

In [6]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ls = lner.LabelSpace(open(f"{DATA}/labels.txt").read().split())
m = lner_ret.RetNER(lm, ls.n).to(dev).train()
x = torch.randint(0, cfg["vocab_size"], (8, 1024), device=dev)
for i in range(3):
    t0 = time.time()
    with torch.autocast("cuda", enabled=dev.type == "cuda"): y = m(x)
    y.float().mean().backward(); torch.cuda.synchronize() if dev.type == "cuda" else None
    print(f"fwd+bwd bs8 x 1024 tokens: {time.time()-t0:.2f}s")
if dev.type == "cuda": print("peak GB:", torch.cuda.max_memory_allocated() / 2**30)
del m, x, y; torch.cuda.empty_cache() if dev.type == "cuda" else None
pc = lner.Piecer(tok); T1 = json.load(open(f"{DATA}/fold1-text.json")); d0 = next(iter(T1.values()))
ids, cs, ce = pc.encode_doc(d0[:600]); print("tokens for first 600 chars:", int((cs >= 0).sum()), "| EOS id used:", lner.EOS_ID, "| PAD:", lner.PAD_ID)

fwd+bwd bs8 x 1024 tokens: 1.83s
fwd+bwd bs8 x 1024 tokens: 0.56s
fwd+bwd bs8 x 1024 tokens: 0.56s
peak GB: 6.885305404663086
tokens for first 600 chars: 146 | EOS id used: 259 | PAD: 256


## 3. Helpers

In [7]:
N_FOLDS = 5
def launch_cv(name, gpu, extra, epochs, folds=None, budget=20, teacher_family=None):
    folds = list(range(N_FOLDS)) if folds is None else folds
    cmds = []
    for k in folds:
        out = f"{WORK}/runs/{name}_f{k}"; os.makedirs(out, exist_ok=True)
        t = f"--teacher {WORK}/runs/{teacher_family}_f{k}/best.pt" if teacher_family else ""
        cmds.append(f"PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} python lner_ret.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {out} "
                    f"--n_folds {N_FOLDS} --fold {k} --epochs {epochs} --time_budget_min {budget} {t} {extra} > {out}/log.txt 2>&1")
    return subprocess.Popen(" ; ".join(cmds), shell=True), name

def wait(jobs, every=120):
    while any(p.poll() is None for p, _ in jobs):
        time.sleep(every)
        for p, n in jobs:
            logs = sorted(glob.glob(f"{WORK}/runs/{n}_f*/log.txt"), key=os.path.getmtime)
            tail = open(logs[-1]).read().strip().splitlines()[-1:] if logs else []
            print(f"[{n}] {'running' if p.poll() is None else 'done'} | {logs[-1].split('/')[-2] if logs else ''} | {tail[0][:120] if tail else ''}")
    for n in sorted({n for _, n in jobs}):
        done = len(glob.glob(f"{WORK}/runs/{n}_f*/test_logp.npz"))
        print(f"== {n}: {done}/{N_FOLDS} folds finished" + ("" if done == N_FOLDS else "   <-- check log.txt of the missing fold"))

def fam_table(names):
    rows = []
    for n in names:
        for d in sorted(glob.glob(f"{WORK}/runs/{n}_f*/results.json")):
            x = json.load(open(d)); rows.append(dict(run=d.split("/")[-2], dev_F1=round(x["dev_macro_F1"], 4), best_step=x["best_step"], steps=x["steps"],
                                                     train_min=round(x["train_minutes"], 1), n_dev_docs=x["n_dev_docs"], trainable_M=round(x["params_trainable"] / 1e6, 2)))
    return pd.DataFrame(rows)

## 4. Smoke test (1 tiny fold, ~3 min). Check for `dev macro` lines and `dev_logp.npz`.

In [8]:
p, n = launch_cv("smoke", 0, "--method lora --max_train_docs 20 --no_test", 1, folds=[0], budget=6)
p.wait(); print("".join(l for l in open(f"{WORK}/runs/smoke_f0/log.txt") if any(k in l for k in ["train docs", "ORACLE", "params", "speed", "eval", "dev macro", "Traceback", "Error"])))
print(os.listdir(f"{WORK}/runs/smoke_f0"))

  train docs=20 dev docs=14 test docs=35
  ORACLE strict macro-F1 of the tokenisation/alignment pipeline: train=0.9950 dev=0.9983
  params total=76.21M trainable=2.646M (3.47%)
  speed=1.47 steps/s -> total steps=60 (planned 42); eval every 50
  [eval mid] step=50 dev strict macro-F1=0.1650 P=0.1524 R=0.2002
  [eval final] step=60 dev strict macro-F1=0.2109 P=0.2208 R=0.2481
  dev macro-F1: greedy=0.1504 viterbi=0.2109 -> use viterbi

['dev_truth.json', 'log.txt', 'best.pt', 'dev_pred.json', 'results.json', 'dev_logp.npz']


## 5. YOUR KNOBS (untuned starting values — hyper-parameter choices must be yours)
`BUDGET` = max minutes per fold (hard stop; the best-per-epoch checkpoint is kept). If `best_step` is always the last step, increase epochs.

In [9]:
EP = dict(lora=10, lorma=10, full=6, kd=6)      # <- change
RANK = 16
BUDGET = dict(lora=18, lorma=18, full=16, kd=16)

## 6. Round A — LoRA (GPU0) ‖ LoRMA₊ (GPU1)

In [10]:
jobs = [launch_cv("ret_lora",  0, f"--method lora  --rank {RANK} --alpha {2*RANK}", EP["lora"],  budget=BUDGET["lora"]),
        launch_cv("ret_lorma", 1, f"--method lorma --rank {RANK} --alpha {RANK}",   EP["lorma"], budget=BUDGET["lorma"])]
wait(jobs); print(fam_table(["ret_lora", "ret_lorma"]).to_string(index=False))

[ret_lora] running | ret_lora_f0 |   [eval mid] step=110 dev strict macro-F1=0.3002 P=0.2800 R=0.3414
[ret_lorma] running | ret_lorma_f0 |   step 150/1100 ep 2 loss 0.4965 lr 9.94e-04 1.7min
[ret_lora] running | ret_lora_f0 |   step 250/1100 ep 3 loss 0.1651 lr 9.46e-04 3.4min
[ret_lorma] running | ret_lorma_f0 |   [eval mid] step=330 dev strict macro-F1=0.3811 P=0.3541 R=0.4259
[ret_lora] running | ret_lora_f0 |   step 400/1100 ep 4 loss 0.0692 lr 7.95e-04 5.5min
[ret_lorma] running | ret_lorma_f0 |   step 500/1100 ep 5 loss 0.0587 lr 6.56e-04 5.8min
[ret_lora] running | ret_lora_f0 |   [eval mid] step=550 dev strict macro-F1=0.4198 P=0.4176 R=0.4290
[ret_lorma] running | ret_lorma_f0 |   [eval mid] step=660 dev strict macro-F1=0.3878 P=0.3903 R=0.4149
[ret_lora] running | ret_lora_f0 |   step 700/1100 ep 7 loss 0.0137 lr 3.47e-04 9.7min
[ret_lorma] running | ret_lorma_f0 |   step 850/1100 ep 8 loss 0.0127 lr 1.48e-04 9.9min
[ret_lora] running | ret_lora_f0 |   step 850/1100 ep 8 loss

## 7. Round B — full fine-tuning (folds split across both GPUs; also the KD teacher)

In [11]:
jobs = [launch_cv("ret_full", 0, "--method full", EP["full"], folds=[0, 1, 2], budget=BUDGET["full"]),
        launch_cv("ret_full", 1, "--method full", EP["full"], folds=[3, 4],    budget=BUDGET["full"])]
wait(jobs); print(fam_table(["ret_full"]).to_string(index=False))

[ret_full] running | ret_full_f0 |   step 150/660 ep 2 loss 0.4383 lr 9.81e-05 1.9min
[ret_full] running | ret_full_f0 |   step 150/660 ep 2 loss 0.4383 lr 9.81e-05 1.9min
[ret_full] running | ret_full_f3 |   step 350/660 ep 4 loss 0.0537 lr 5.87e-05 3.8min
[ret_full] running | ret_full_f3 |   step 350/660 ep 4 loss 0.0537 lr 5.87e-05 3.8min
[ret_full] running | ret_full_f0 |   step 450/660 ep 5 loss 0.0236 lr 3.11e-05 5.7min
[ret_full] running | ret_full_f0 |   step 450/660 ep 5 loss 0.0236 lr 3.11e-05 5.7min
[ret_full] running | ret_full_f4 |   windows/epoch=920 steps/epoch=115 planned steps=690
[ret_full] running | ret_full_f4 |   windows/epoch=920 steps/epoch=115 planned steps=690
[ret_full] running | ret_full_f4 |   step 150/690 ep 2 loss 0.4513 lr 9.83e-05 1.7min
[ret_full] running | ret_full_f4 |   step 150/690 ep 2 loss 0.4513 lr 9.83e-05 1.7min
[ret_full] running | ret_full_f4 |   step 350/690 ep 4 loss 0.0585 lr 6.21e-05 4.0min
[ret_full] running | ret_full_f4 |   step 350/69

## 8. Round C — KD: 6-layer student (teacher blocks 0,2,4,7,9,11) distilled from the SAME fold's full model (no leakage)

In [12]:
jobs = [launch_cv("ret_kd", 0, "--method kd --student_layers 6", EP["kd"], folds=[0, 1, 2], budget=BUDGET["kd"], teacher_family="ret_full"),
        launch_cv("ret_kd", 1, "--method kd --student_layers 6", EP["kd"], folds=[3, 4],    budget=BUDGET["kd"], teacher_family="ret_full")]
wait(jobs); print(fam_table(["ret_kd"]).to_string(index=False))

[ret_kd] running | ret_kd_f3 |   [eval mid] step=220 dev strict macro-F1=0.3273 P=0.3610 R=0.3272
[ret_kd] running | ret_kd_f3 |   [eval mid] step=220 dev strict macro-F1=0.3273 P=0.3610 R=0.3272
[ret_kd] running | ret_kd_f3 |   step 450/660 ep 5 loss 0.0652 lr 3.11e-05 3.9min
[ret_kd] running | ret_kd_f3 |   step 450/660 ep 5 loss 0.0652 lr 3.11e-05 3.9min
[ret_kd] running | ret_kd_f3 |     CASENO     P=0.412 R=0.400 F1=0.406  (tp=84 fp=120 fn=126)
[ret_kd] running | ret_kd_f3 |     CASENO     P=0.412 R=0.400 F1=0.406  (tp=84 fp=120 fn=126)
[ret_kd] running | ret_kd_f1 |   step 50/672 ep 1 loss 2.0322 lr 5.00e-05 0.5min
[ret_kd] running | ret_kd_f1 |   step 50/672 ep 1 loss 2.0322 lr 5.00e-05 0.5min
[ret_kd] running | ret_kd_f4 |   step 400/690 ep 4 loss 0.0838 lr 4.89e-05 3.6min
[ret_kd] running | ret_kd_f4 |   step 400/690 ep 4 loss 0.0838 lr 4.89e-05 3.6min
[ret_kd] running | ret_kd_f4 |   step 600/690 ep 6 loss 0.0338 lr 5.76e-06 5.4min
[ret_kd] running | ret_kd_f4 |   step 600/69

## 9. OOF scores + ensembles

In [13]:
os.makedirs(f"{WORK}/submissions_ret", exist_ok=True)
def run_ens(out, names, extra=""):
    r = subprocess.run(f"python ensemble.py --data_dir {DATA} --runs_dir {WORK}/runs --names {names} --out {WORK}/ens/{out} {extra}", shell=True, capture_output=True, text=True)
    print(r.stdout[-4500:], r.stderr[-1500:])
    z = f"{WORK}/ens/{out}/submission.zip"
    if os.path.exists(z): shutil.copy(z, f"{WORK}/submissions_ret/{out}.zip")

ALL = [n for n in ["ret_lora", "ret_lorma", "ret_full", "ret_kd"] if len(glob.glob(f"{WORK}/runs/{n}_f*/test_logp.npz")) == N_FOLDS]
print("complete families:", ALL)
for n in ALL: run_ens(f"ens_{n}", n)

complete families: ['ret_lora', 'ret_lorma', 'ret_full', 'ret_kd']
  token cache: /kaggle/working/p2data_ret/tokcache_8f7c4275df.pkl
  family ret_lora: 5 runs
  ret_lora: OOF coverage 70/70 docs

OUT-OF-FOLD strict macro-F1 on 70 labelled docs (viterbi, no post-processing)
  ret_lora       F1=0.4192  P=0.4235  R=0.4222
  ENSEMBLE       F1=0.4192  P=0.4235  R=0.4222

chosen: min_conf=0.0 consistency=0 -> OOF macro-F1 0.4192
    APP        P=0.112 R=0.074 F1=0.089
    RESP       P=0.236 R=0.108 F1=0.148
    A.COUNSEL  P=0.451 R=0.400 F1=0.424
    R.COUNSEL  P=0.418 R=0.511 F1=0.460
    JUDGE      P=0.481 R=0.531 F1=0.505
    WIT        P=0.126 R=0.123 F1=0.125
    AUTH       P=0.335 R=0.276 F1=0.303
    COURT      P=0.748 R=0.775 F1=0.761
    STAT       P=0.642 R=0.599 F1=0.620
    PREC       P=0.280 R=0.340 F1=0.307
    DATE       P=0.879 R=0.907 F1=0.893
    CASENO     P=0.375 R=0.424 F1=0.398
wrote /kaggle/working/ens/ens_ret_lora/submission.zip from 5 runs (3860 entities in 35 docs)


In [14]:
run_ens("ens_ret_all_grid", ",".join(ALL), "--grid")

  token cache: /kaggle/working/p2data_ret/tokcache_8f7c4275df.pkl
  family ret_lora: 5 runs
  family ret_lorma: 5 runs
  family ret_full: 5 runs
  family ret_kd: 5 runs
  ret_lora: OOF coverage 70/70 docs
  ret_lorma: OOF coverage 70/70 docs
  ret_full: OOF coverage 70/70 docs
  ret_kd: OOF coverage 70/70 docs

OUT-OF-FOLD strict macro-F1 on 70 labelled docs (viterbi, no post-processing)
  ret_lora       F1=0.4192  P=0.4235  R=0.4222
  ret_lorma      F1=0.4051  P=0.4019  R=0.4147
  ret_full       F1=0.4068  P=0.4097  R=0.4123
  ret_kd         F1=0.3660  P=0.3815  R=0.3605
  ENSEMBLE       F1=0.4360  P=0.4612  R=0.4238

Post-processing grid on the OOF ensemble (min_conf x consistency)  -> macro-F1 / P / R
  min_conf=0.0 consistency=0  F1=0.4360 P=0.4612 R=0.4238
  min_conf=0.0 consistency=1  F1=0.4391 P=0.4482 R=0.4402
  min_conf=0.3 consistency=0  F1=0.4375 P=0.4650 R=0.4238
  min_conf=0.3 consistency=1  F1=0.4412 P=0.4529 R=0.4407
  min_conf=0.4 consistency=0  F1=0.4390 P=0.4740 R=0.4

In [15]:
MIN_CONF, CONSIST = 0.0, 0                      # <- set from the grid above
run_ens("ens_ret_all_final", ",".join(ALL), f"--min_conf {MIN_CONF} --consistency {CONSIST}")
print(sorted(os.listdir(f"{WORK}/submissions_ret")))

  token cache: /kaggle/working/p2data_ret/tokcache_8f7c4275df.pkl
  family ret_lora: 5 runs
  family ret_lorma: 5 runs
  family ret_full: 5 runs
  family ret_kd: 5 runs
  ret_lora: OOF coverage 70/70 docs
  ret_lorma: OOF coverage 70/70 docs
  ret_full: OOF coverage 70/70 docs
  ret_kd: OOF coverage 70/70 docs

OUT-OF-FOLD strict macro-F1 on 70 labelled docs (viterbi, no post-processing)
  ret_lora       F1=0.4192  P=0.4235  R=0.4222
  ret_lorma      F1=0.4051  P=0.4019  R=0.4147
  ret_full       F1=0.4068  P=0.4097  R=0.4123
  ret_kd         F1=0.3660  P=0.3815  R=0.3605
  ENSEMBLE       F1=0.4360  P=0.4612  R=0.4238

chosen: min_conf=0.0 consistency=0 -> OOF macro-F1 0.4360
    APP        P=0.175 R=0.081 F1=0.111
    RESP       P=0.294 R=0.118 F1=0.169
    A.COUNSEL  P=0.446 R=0.421 F1=0.433
    R.COUNSEL  P=0.474 R=0.505 F1=0.489
    JUDGE      P=0.506 R=0.510 F1=0.508
    WIT        P=0.153 R=0.096 F1=0.118
    AUTH       P=0.361 R=0.233 F1=0.283
    COURT      P=0.770 R=0.768 F1=0

## 10. Report tables (method comparison and per-label OOF F1

In [16]:
rows, per_label = [], {}
for n in ALL:
    rs = [json.load(open(f)) for f in sorted(glob.glob(f"{WORK}/runs/{n}_f*/results.json"))]
    rep = json.load(open(f"{WORK}/ens/ens_{n}/ensemble_report.json"))
    rows.append(dict(method=n, OOF_macro_F1=round(rep["oof_ensemble_F1"], 4), mean_fold_dev_F1=round(np.mean([r["dev_macro_F1"] for r in rs]), 4),
                     params_total_M=round(rs[0]["params_total"] / 1e6, 1), trainable_M=round(rs[0]["params_trainable"] / 1e6, 3),
                     trainable_pct=round(100 * rs[0]["params_trainable"] / rs[0]["params_total"], 2), train_min_per_fold=round(np.mean([r["train_minutes"] for r in rs]), 1),
                     peak_gpu_GB=round(np.mean([r["peak_gpu_gb"] for r in rs]), 2), infer_tok_per_s=int(np.mean([r["infer_tokens_per_s"] for r in rs]))))
    per_label[n] = {l: round(v["F1"], 3) for l, v in rep["per_label"].items()}
summary = pd.DataFrame(rows); PL = pd.DataFrame(per_label)
print(summary.to_string(index=False)); print(); print(PL.to_string())
summary.to_csv(f"{WORK}/ret_summary.csv", index=False); PL.to_csv(f"{WORK}/ret_per_label_oof_f1.csv")
NUMK = ["exact", "boundary", "label_confusion", "spurious", "missed"]
eb = {n: pd.DataFrame([{k: json.load(open(f))["errors"][k] for k in NUMK} for f in sorted(glob.glob(f"{WORK}/runs/{n}_f*/results.json"))]).sum() for n in ALL}
print(); print(pd.DataFrame(eb).to_string())

   method  OOF_macro_F1  mean_fold_dev_F1  params_total_M  trainable_M  trainable_pct  train_min_per_fold  peak_gpu_GB  infer_tok_per_s
 ret_lora        0.4192            0.4209            76.2        2.646           3.47                15.3         6.88            22377
ret_lorma        0.4051            0.4082            76.4        2.867           3.75                13.2         6.53            25862
 ret_full        0.4068            0.4084            73.6       73.583         100.00                 8.0         7.43            29440
   ret_kd        0.3660            0.3666            40.6       40.640         100.00                 6.4         4.37            58931

           ret_lora  ret_lorma  ret_full  ret_kd
APP           0.089      0.086     0.103   0.070
RESP          0.148      0.154     0.147   0.133
A.COUNSEL     0.424      0.441     0.421   0.349
R.COUNSEL     0.460      0.453     0.429   0.307
JUDGE         0.505      0.461     0.428   0.410
WIT           0.125      

## 11. (Optional) Mix with the decoder runs
Only valid if the decoder and the retention model used the **same tokenizer file** (else the token streams differ and `ensemble.py` stops with a clear message).
Attach the decoder notebook's output as a dataset, point `DEC_RUNS` at its `runs` folder, and the cell copies those families next to the retention ones.

In [17]:
DEC_RUNS = None        # e.g. "/kaggle/input/<decoder-output>/runs"
if DEC_RUNS:
    for d in glob.glob(f"{DEC_RUNS}/*_f[0-9]*"):
        shutil.copytree(d, f"{WORK}/runs/dec_{os.path.basename(d)}", dirs_exist_ok=True)
    DEC = sorted({os.path.basename(d).rsplit("_f", 1)[0] for d in glob.glob(f"{WORK}/runs/dec_*_f[0-9]*")})
    print("decoder families:", DEC)
    run_ens("ens_dec_plus_ret", ",".join(DEC + ALL), "--grid")

Upload every zip in `submissions_ret/` to Codabench. For the report: the OOF tables, Codabench scores, and model comparison (accuracy, parameters, speed, memory).